In [1]:
# Upgraded Project Improvements: 
    # Model Architecture Upgraded
        # The baseline Random Forest classifier was replaced with a feedforward neural network (MLP), enabling nonlinear feature learning and improved representation capacity.
    # Robust Evaluation Framework
        # The model is evaluated using 50 Monte Carlo train-test splits, with full model re-initialization for each run.
        # This provides averaged class-specific performance metrics (accuracy, precision, recall, F1-score) for both pathogenic and benign classes.
        # This approach reduces dependence on a single train-test split and improves estimate stability.
    # Enhanced Feature Engineering
        # The feature set was expanded to include biochemical properties of amino acids (charge, polarity, and size).
        # This enables the model to learn biologically meaningful relationships between mutation properties and pathogenicity.

In [2]:
# Loads the pandas library
# Allows to use pd as an alias instead of typing pandas every time we want to use it
import pandas as pd

# Imports Python's modern path-handling system 
from pathlib import Path

# Defines a function to automatically find the main project folder 
def find_root():
    # Sets variable p to current working directory (Ex: Notebooks/01_MutationProj.ipynb)
    p = Path.cwd()
    # Loop that continues moving upward through parent directories until top of filesystem is reached (Cancer-Mutation-Classification-TensorFlow)
    while p != p.parent:
        # Checks to see if file .projectroot exists inside p directory by appending .projectroot to p
        # If found, returns the directory containing projectroot 
        if (p / ".projectroot").exists():
            return p
        # If projectroot not found, moves up one folder and continues loop again 
        p = p.parent
    # Searched to the top of the filesystem without finding project root
    raise FileNotFoundError("Project root not found")

# Sets ROOT to the directory containing .projectroot (Cancer-Mutation-Classification-TensorFlow) 
ROOT = find_root()

# Appends "Datasets" folder to the root (Ex: Cancer-Mutation-Classification-TensorFlow/Datasets)
DATA_DIR = ROOT / "Datasets"

# Creates a Path object pointing to the file location (Cancer-Mutation-Classification-TensorFlow/Datasets/Data_TP53.txt)
tp53_path = DATA_DIR / "Data_TP53.txt"

# Reads the file located at tp_53 path and converts it into a pandas DataFrame
# Tells pandas that the columns in the file are tab-separated (sep="\t")
tp53_df = pd.read_csv(tp53_path, sep="\t")

# Splits the "Name" column into three new columns: "ref_aa", "position", and "mut_aa" based on string 
tp53_df[["ref_aa", "position", "mut_aa"]] = tp53_df["Name"].str.extract(
    r"p\.([A-Za-z]+)(\d+)([A-Za-z]+)"
)

# Defines a function that takes in a string, returns either Benign or Pathogenic
# Converts "Likely Pathogenic" classification into Pathogenic Classification
# Converts "Likely Benign" classification into Benign Classification
def clean_label(x):
     x = str(x).lower()
     if "benign" in x and "pathogenic" not in x:
         return "Benign"
     elif "pathogenic" in x:
         return "Pathogenic"
     else:
         return "Other"
     
# Creates a new column "Classification"
# Sets it equal to the Germline classification column after applying the clean_label function to it
tp53_df["Classification"] = tp53_df["Germline classification"].apply(clean_label)

In [3]:
# Creates a new DataFrame called new_df that only contains the columns mentioned below from the original DataFrame df
# .copy() prevents changes to new_df that could affect original_df data
new_df = tp53_df[["ref_aa", "position", "mut_aa", "Classification"]].copy()

# Displays the first five rows of new_df
pd.set_option('display.max_rows', None)
new_df.head()

,ref_aa,position,mut_aa,Classification
0,Asp,393,Tyr,Benign
1,Glu,388,Asp,Benign
2,Glu,388,Asp,Benign
3,Glu,388,Ala,Benign
4,Glu,388,Gln,Benign


In [4]:
# Creates a frequency table of Classification labels with counts per for Benign and Pathogenic for Tp53 gene
new_df["Classification"].value_counts().rename_axis("Class").reset_index(name="Count")

,Class,Count
0,Pathogenic,244
1,Benign,149


In [5]:
# Creates a list that:
# 1. Joins both the ref_aa and mut_aa columns
# 2. Removes duplicates using set()
# 3. Sorts the list alphabetically.

# Creates a dictionary that maps each amino acid to a unique index based on its position in the sorted list
aa_list = sorted(set(new_df["ref_aa"]).union(set(new_df["mut_aa"])))
aa_to_idx = {aa: i for i, aa in enumerate(aa_list)}

In [6]:
# Imports the numpy library, uses the alias np so that it can be used instead of typing numpy every time
import numpy as np

# Created an empty list X 
X = []

# Iterates through each row of the new_df DataFrame using iterrows()
# Extract values for ref, mut, pos, and appends them as a list to X
# Ex: [3, 4, 175]
for _, row in new_df.iterrows():
    ref = aa_to_idx[row["ref_aa"]]
    mut = aa_to_idx[row["mut_aa"]]
    pos = int(row["position"])

    X.append([ref, mut, pos])

# Created a numpy array from the list X, turns it into an efficient numerical matrix 
X = np.array(X)

In [7]:
# Creates a panda series that turns the classification column into a binary variable
# .apply() is a panda features that iterates through each row by row  
y = new_df["Classification"].apply(
    lambda x: 1 if "pathogenic" in x.lower() else 0
)

# Created a numpy array from the panda series y, turns it into an efficient numerical vector
y = np.array(y)

In [8]:
# Created a nested dictionary that contains the properties of each amino acid, such as charge, polarity, hydrophobicity, and size
# Negative charge represented as -1, positive charge represented as 1, and neutral charge represented as 0
# Polarity encoded numerically, represented as positive for polar amino acids and negative for non-polar amino acids
# Size represented amino acid r-chain size, with larger numbers indicating larger amino acids
aa_properties = {
    "Ala": {"charge": 0, "polarity": 0, "hydrophobicity": 1.8, "size": 1},
    "Arg": {"charge": 1, "polarity": 1, "hydrophobicity": -4.5, "size": 3},
    "Asn": {"charge": 0, "polarity": 1, "hydrophobicity": -3.5, "size": 2},
    "Asp": {"charge": -1, "polarity": 1, "hydrophobicity": -3.5, "size": 2},
    "Cys": {"charge": 0, "polarity": 0, "hydrophobicity": 2.5, "size": 2},
    "Gln": {"charge": 0, "polarity": 1, "hydrophobicity": -3.5, "size": 3},
    "Glu": {"charge": -1, "polarity": 1, "hydrophobicity": -3.5, "size": 3},
    "Gly": {"charge": 0, "polarity": 0, "hydrophobicity": -0.4, "size": 1},
    "His": {"charge": 1, "polarity": 1, "hydrophobicity": -3.2, "size": 3},
    "Ile": {"charge": 0, "polarity": 0, "hydrophobicity": 4.5, "size": 3},
    "Leu": {"charge": 0, "polarity": 0, "hydrophobicity": 3.8, "size": 3},
    "Lys": {"charge": 1, "polarity": 1, "hydrophobicity": -3.9, "size": 3},
    "Met": {"charge": 0, "polarity": 0, "hydrophobicity": 1.9, "size": 3},
    "Phe": {"charge": 0, "polarity": 0, "hydrophobicity": 2.8, "size": 3},
    "Pro": {"charge": 0, "polarity": 0, "hydrophobicity": -1.6, "size": 2},
    "Ser": {"charge": 0, "polarity": 1, "hydrophobicity": -0.8, "size": 2},
    "Thr": {"charge": 0, "polarity": 1, "hydrophobicity": -0.7, "size": 2},
    "Trp": {"charge": 0, "polarity": 0, "hydrophobicity": -0.9, "size": 4},
    "Tyr": {"charge": 0, "polarity": 1, "hydrophobicity": -1.3, "size": 3},
    "Val": {"charge": 0, "polarity": 0, "hydrophobicity": 4.2, "size": 2}
}

In [9]:
import numpy as np

# Creates empty lists X and y to store the features and labels for the machine learning model
X = []
y = []

# Iterates through each row of the new_df DataFrame using iterrows()
for _, row in new_df.iterrows():
    ref = row["ref_aa"]
    mut = row["mut_aa"]

    # Checks if the reference and mutant amino acids are present in the aa_properties dictionary, if not, it skips to the next iteration
    if ref not in aa_properties or mut not in aa_properties:
        continue

    # Retrieves the physicochemical properties of the reference and mutant amino acids
    ref_p = aa_properties[ref]
    mut_p = aa_properties[mut]

    # Appends a list of features to X for each mutation, including the position of the mutation and the differences in properties between the mutant and reference amino acids
    X.append([
        int(row["position"]),
        mut_p["charge"] - ref_p["charge"],
        mut_p["polarity"] - ref_p["polarity"],
        mut_p["hydrophobicity"] - ref_p["hydrophobicity"],
        mut_p["size"] - ref_p["size"]
    ])

    # Appends a binary label to y for each mutation, where 1 indicates pathogenic and 0 indicates benign
    label = row["Classification"]
    y.append(1 if "pathogenic" in str(label).lower() else 0)

# Converts the lists X and y into numpy arrays of type float32, which are more efficient for numerical computations in machine learning models
X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

In [10]:
# Imports the tensorflow library, uses the alias tf so that it can be used instead of typing tensorflow every time
import tensorflow as tf

def build_model():

    # Sequential model is a linear stack of layers, allows us to build a neural network by adding layers one after the other
    model = tf.keras.Sequential([

        # Tells the model how many features each input sample has
        tf.keras.Input(shape=(X_train.shape[1],)),

        # First layer is a Dense layer with 16 neurons, ReLU activation function (introduces non-linearity), and input shape equal to the number of features in X_train
        tf.keras.layers.Dense(16, activation='relu'),

        # Second layer is a Dense layer with 8 neurons and ReLU activation function (introduces non-linearity)
        # Refines patterns learned from first layer
        tf.keras.layers.Dense(8, activation='relu'),

        # Final layer is a Dense layer with 1 neuron and sigmoid activation function (outputs a value between 0 and 1, representing the probability of being pathogenic)
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    
    # Configures how the model will learn, tells network how update itself and how to measure performance
    # Adam controls how the model updates its weights during training
    # Binary crossentropy is used for binary classification problems, measures the difference between predicted probabilities and actual labels
    # Accuracy is used as a metric to evaluate the performance of the model during training and testing
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [11]:
import numpy as np

# Imports functions from scikit-learn library that allow us to evaluate the performance of our model
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

# Imports a function from scikit-learn library that allows us to split our data into training and testing sets
from sklearn.model_selection import train_test_split

accuracies = []
f1_scores_pathogenic = []
f1_scores_benign = []
recall_scores_pathogenic = []
recall_scores_benign = []
precision_scores_pathogenic = []
precision_scores_benign = []

for i in range(50):

    # 20% of the data is reserved for testing (X_test, y_test), 80% is used for training (x_train, y_train)
    # Use a different random_state for each iteration so each run evaluates the model on a different stratified train/test split
    X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=i
    )

    # Controls NumPy's randomness
    np.random.seed(42)
    
    # Controls TensorFlow's randomness 
    tf.random.set_seed(42)

    # 1. Rebuild model each time
    model = build_model()

    # 2. Trains the model on the training data for 10 epochs (iterations over the entire dataset)
    # Batch size of 16 means the model will update its weights after processing every 16 samples within each epoch 
    # Validation split: Within X_train (which is 80% of the full dataset), 80% is used to train the model (64% of total data), and 20% is set aside for validation (16% of total data)
    # Validation Accuracy is accuracy on unseen-in-training data
        # Goes up after each epoch, signifies that the model is in fact learning useful patterns
    # Validation data is what the model uses to evaluate its performance after each epoch (accuracy, loss, val_accuracy, val_loss)
    # Overfitting: Too many epochs can decrease accuracy because the model begins to memorize training data which doesn't generalize to the validation data
    model.fit(X_train, y_train, epochs=10, batch_size=16, verbose=0)

    # 3. Model outputs a probability
    # Probability greater than 0.5, than predicts test sample as pathogenic (1)
    # If probability is less than 0.5 than predicts test sample as benign (0)
    y_prob = model.predict(X_test, verbose=0)   
    y_pred = (y_prob > 0.5).astype(int)

    # 4. Calculates both Pathogenic and Benign accuracy, f1 score, recall, and precision for each iteration
    # pos_label=1 indicates pathogenic
    # pos_label=0 indicates benign 
    acc = accuracy_score(y_test, y_pred)
    f1_pathogenic = f1_score(y_test, y_pred, pos_label=1)
    f1_benign = f1_score(y_test, y_pred,pos_label=0)
    recall_pathogenic = recall_score(y_test, y_pred, pos_label=1)
    recall_benign = recall_score(y_test, y_pred, pos_label=0)
    precision_pathogenic = precision_score(y_test, y_pred, pos_label=1)
    precision_benign = precision_score(y_test, y_pred, pos_label=0)
    accuracies.append(acc)

    # Appends the 50 scores to each of the lists for each metric 
    f1_scores_pathogenic.append(f1_pathogenic)
    f1_scores_benign.append(f1_benign)
    recall_scores_pathogenic.append(recall_pathogenic)
    recall_scores_benign.append(recall_benign)
    precision_scores_pathogenic.append(precision_pathogenic)
    precision_scores_benign.append(precision_benign)

# Prints the classification metrics for the model, averages the scores across all 50 iterations
print("Accuracy:", np.mean(accuracies), "\n")
print("Recall (Pathogenic):", np.mean(recall_scores_pathogenic))
print("Recall (Benign):", np.mean(recall_scores_benign), "\n")
print("Precision (Pathogenic):", np.mean(precision_scores_pathogenic))
print("Precision (Benign):", np.mean(precision_scores_benign), "\n")
print("F1 Score (Pathogenic):", np.mean(f1_scores_pathogenic))
print("F1 Score (Benign):", np.mean(f1_scores_benign))

c:\Users\Tristan\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


c:\Users\Tristan\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Tristan\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Tristan\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"

Accuracy: 0.5908860759493672 

Recall (Pathogenic): 0.7995918367346938
Recall (Benign): 0.25000000000000006 

Precision (Pathogenic): 0.6208970722882582
Precision (Benign): 0.5421512638214534 

F1 Score (Pathogenic): 0.6711584893368755
F1 Score (Benign): 0.24893994478232304


In [12]:
# Upgraded Project Main Limitations:
    # Dataset Scope
        # The model was trained exclusively on TP53 mutation data (393 samples), limiting its ability to generalize to other genes or broader mutational contexts.
    # Class Imbalance
        # The dataset contains an imbalance between pathogenic (244) and benign (149) samples, which may bias predictions toward the majority class.
    # Residual Variance in Evaluation
        # Despite using 50 train-test splits, performance metrics still vary across runs due to stochastic training, random train-test splitting, and small dataset size.
    # Limited Test Sample Size per Run
        # Each evaluation still uses relatively small test subsets (e.g., 79 samples), resulting in high variance in performance estimates.
    # Sample Size and Evaluation Stability
        # The dataset size is relatively small, particularly the test set (79 samples), which increases variance in performance estimates.
        # A single train-test split was used, meaning results may vary depending on data partitioning.
        # Cross-validation was not performed, which limits robustness of reported metrics.
    # Degenerate Predictions in Some Runs
        # In certain iterations, the model collapses to predicting a single class, resulting in undefined precision/recall for the minority class and warnings
        # "Precision is ill-defined and being set to 0.0 due to no predicted samples"
    # Lack of Statistical Uncertainty Quantification
        # Performance metrics are reported without confidence intervals, limiting formal statistical inference regarding model stability and reliability